In [1]:
import pandas as pd

# 1. Load the CSV into a DataFrame
df = pd.read_csv('data/netflix_titles.csv')

print("--- BEFORE CLEANING ---")
print(f"Total rows and columns: {df.shape}")
print("Missing values per column:")
print(df.isnull().sum())

# 2. Drop rows ONLY if 'title' or 'description' is missing
df = df.dropna(subset=['title', 'description'])

# 3. Fill missing values in other columns with an empty string ''
df['director'] = df['director'].fillna('')
df['cast'] = df['cast'].fillna('')
df['country'] = df['country'].fillna('')
df['rating'] = df['rating'].fillna('')

print("\n--- AFTER CLEANING ---")
print(f"Total rows and columns: {df.shape}")
print("Missing values per column:")
print(df[['title', 'director', 'cast', 'description']].isnull().sum())

--- BEFORE CLEANING ---
Total rows and columns: (8807, 12)
Missing values per column:
show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       0
rating             4
duration           3
listed_in          0
description        0
dtype: int64

--- AFTER CLEANING ---
Total rows and columns: (8807, 12)
Missing values per column:
title          0
director       0
cast           0
description    0
dtype: int64


In [2]:
import re

def tok(text, limit):
    if not text:
        return ''
    items = [i.strip() for i in text.split(',')][:limit]
    return ' '.join(re.sub(r'[^a-z0-9]', '', i.lower()) for i in items)

def tok_genre(text):
    out = []
    for g in text.split(','):
        g = g.strip().lower()
        g = re.sub(r'^tv\s+', '', g)                  # "TV Horror" -> "horror"
        g = re.sub(r'\s+(movies|tv shows)$', '', g)   # "Horror Movies" -> "horror"
        out.append(re.sub(r'[^a-z0-9]', '', g))
    return ' '.join(out)

df['duration'] = df['duration'].fillna('')
df['people_soup'] = (
    df['director'].apply(lambda x: tok(x, 1)) + ' ' +
    df['cast'].apply(lambda x: tok(x, 3))
)
df['genre_soup'] = df['listed_in'].apply(tok_genre)
df['desc_soup'] = df['description'].str.lower()

In [3]:
import numpy as np
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

df = df.reset_index(drop=True)

tfidf_desc   = TfidfVectorizer(stop_words='english', min_df=2, sublinear_tf=True)
tfidf_genre  = TfidfVectorizer(token_pattern=r'\S+')
tfidf_people = TfidfVectorizer(token_pattern=r'\S+', min_df=2)

desc_matrix   = tfidf_desc.fit_transform(df['desc_soup'])
genre_matrix  = tfidf_genre.fit_transform(df['genre_soup'])
people_matrix = tfidf_people.fit_transform(df['people_soup'])

# Tunable weights (should sum to 1)
W_DESC, W_GENRE, W_PEOPLE = 0.45, 0.40, 0.15

tfidf_matrix = hstack([
    np.sqrt(W_DESC)   * desc_matrix,
    np.sqrt(W_GENRE)  * genre_matrix,
    np.sqrt(W_PEOPLE) * people_matrix,
]).tocsr()
print(f"Combined matrix shape: {tfidf_matrix.shape}")

Combined matrix shape: (8807, 14541)


In [4]:
genres = df['listed_in'].apply(lambda s: set(tok_genre(s).split()))

def avg_genre_overlap(n=300, k=5, seed=0):
    rng = np.random.default_rng(seed)
    scores = []
    for idx in rng.choice(len(df), n, replace=False):
        sims = cosine_similarity(tfidf_matrix[idx], tfidf_matrix).ravel()
        for j in np.argsort(-sims)[1:k+1]:
            a, b = genres[idx], genres[j]
            scores.append(len(a & b) / len(a | b))
    return np.mean(scores)

print(avg_genre_overlap())

0.9564111111111112


In [5]:
def recommend(movie_title):
    # 1. Check if the movie exists in our dataset (case-insensitive)
    matches = df[df['title'].str.lower() == movie_title.lower()]
    if matches.empty:
        print(f"❌ '{movie_title}' not found in the dataset!")
        return

    # Get the exact row index (0 to 8806) of the matching movie
    idx = matches.index[0]

    # 2. Pair each movie index with its similarity score (computed on demand)
    sim_scores = list(enumerate(cosine_similarity(tfidf_matrix[idx], tfidf_matrix).ravel()))

    # 3. Sort from highest score to lowest
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # 4. Skip [0] (the movie itself) and take the top 5 matches
    top_5 = sim_scores[1:6]

    print(f"\n🎬 Top 5 Recommendations for '{df['title'].iloc[idx]}':")
    print("-" * 50)
    for movie_idx, score in top_5:
        title = df['title'].iloc[movie_idx]
        genres = df['listed_in'].iloc[movie_idx]
        print(f"• {title} (Match: {score*100:.1f}%) | Genres: {genres}")

# Test on 3 very different titles
recommend("Stranger Things")
recommend("Inception")
recommend("Naruto")


🎬 Top 5 Recommendations for 'Stranger Things':
--------------------------------------------------
• Helix (Match: 43.4%) | Genres: TV Horror, TV Mysteries, TV Sci-Fi & Fantasy
• Chilling Adventures of Sabrina (Match: 40.0%) | Genres: TV Horror, TV Mysteries, TV Sci-Fi & Fantasy
• Nightflyers (Match: 40.0%) | Genres: TV Horror, TV Mysteries, TV Sci-Fi & Fantasy
• Manifest (Match: 36.7%) | Genres: TV Dramas, TV Mysteries, TV Sci-Fi & Fantasy
• The OA (Match: 36.6%) | Genres: TV Dramas, TV Mysteries, TV Sci-Fi & Fantasy

🎬 Top 5 Recommendations for 'Inception':
--------------------------------------------------
• Dragonheart: A New Beginning (Match: 38.2%) | Genres: Action & Adventure, Sci-Fi & Fantasy
• Alien Warfare (Match: 37.9%) | Genres: Action & Adventure, Sci-Fi & Fantasy
• Level 16 (Match: 37.9%) | Genres: Sci-Fi & Fantasy, Thrillers
• Dark Matter (Match: 37.9%) | Genres: TV Sci-Fi & Fantasy, TV Thrillers
• Star Trek: Deep Space Nine (Match: 37.7%) | Genres: TV Action & Adventure

In [6]:
import pickle
import os

# Create the artifacts folder if it doesn't exist
os.makedirs('artifacts', exist_ok=True)

# Keep only the columns needed for the web app UI to keep the file lightweight
movies_export = df[['title', 'type', 'release_year', 'rating', 'duration', 'listed_in', 'director', 'cast', 'description']]

# Save the dataframe and similarity matrix as binary pickle files
pickle.dump(movies_export, open('artifacts/movies.pkl', 'wb'))
pickle.dump(tfidf_matrix, open('artifacts/tfidf_matrix.pkl', 'wb'))

print("✅ Saved movies.pkl and tfidf_matrix.pkl inside artifacts/!")

✅ Saved movies.pkl and tfidf_matrix.pkl inside artifacts/!
